In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# 1. PARÁMETROS DE LA SIMULACIÓN
# =====================================================================
SEM_CORTE = 20
PLAZO = 52
BAC_TOTAL = 4126638

# Metas globales auditadas para la Semana 20 (CPI 0.82 y SPI 0.86)
TARGET_PV = 1784771
TARGET_EV = 1537173
TARGET_AC = 1874601

# =====================================================================
# 2. DICCIONARIO WBS (122 Actividades de Subestación 115kV)
# =====================================================================
wbs_base = [
    ("1", "1. Ingeniería", "1.1", "1.1 Estudios Previos", ["Levantamiento Topografico", "Estudio de Suelos", "Estudio Resistividad", "Licencia Ambiental", "Aprobacion Trazado", "Estudio Arqueologico"], 1, 6, 0.01),
    ("1", "1. Ingeniería", "1.2", "1.2 Ingeniería Básica", ["Diagrama Unifilar", "Disposicion Planta", "Specs Trafo", "Cortocircuito", "Protecciones", "Basica Control"], 3, 10, 0.01),
    ("1", "1. Ingeniería", "1.3", "1.3 Detalle Civil", ["Cimentacion Trafo", "Porticos 115kV", "Drenajes", "Cajas y Ductos", "Edificio Control", "Memorias Civil", "Vias Acceso", "Cerramiento"], 5, 15, 0.015),
    ("1", "1. Ingeniería", "1.4", "1.4 Detalle Eléctrica", ["Malla a Tierra", "Conexionado", "Servicios Aux", "Control", "Cables", "Alumbrado", "Contra Incendio"], 8, 20, 0.015),

    ("2", "2. Obras Civiles", "2.1", "2.1 Movimiento Tierras", ["Descapote", "Excavacion Terrazas", "Relleno", "Vias Perimetrales", "Taludes", "Cerramiento Prov"], 5, 14, 0.04),
    ("2", "2. Obras Civiles", "2.2", "2.2 Cimentaciones", ["Zapatas Porticos", "Acero Porticos", "Vaciado Porticos", "Foso Trafo", "Muro Cortafuego", "Vaciado Foso", "Trampa Aceite", "Bases GIS"], 10, 25, 0.08),
    ("2", "2. Obras Civiles", "2.3", "2.3 Ductos y Drenajes", ["Excavacion Zanjas", "Tuberia PVC", "Cajas Inspeccion", "Sumideros", "Filtro Frances", "Concreto Ductos", "Relleno Zanjas", "Prueba Hidraulica"], 12, 28, 0.06),
    ("2", "2. Obras Civiles", "2.4", "2.4 Edificio Control", ["Cimentacion Edificio", "Vigas Cimentacion", "Mamposteria", "Losa Cubierta", "Pisos Falsos", "Aire Acondicionado", "Acabados", "Puertas"], 15, 35, 0.12),

    ("3", "3. Suministros", "3.1", "3.1 Equipos 115kV", ["Trafo 30MVA", "Interruptores 115kV", "Seccionadores", "TCs", "TTs", "Pararrayos", "Aisladores Soporte"], 1, 40, 0.25),
    ("3", "3. Suministros", "3.2", "3.2 Estructuras", ["Porticos 115kV", "Vigas Amarre", "Soportes Equipos", "Herrajes", "Pernos Anclaje"], 5, 30, 0.05),
    ("3", "3. Suministros", "3.3", "3.3 Control", ["Tableros Proteccion", "Tableros Control", "Celdas GIS 34.5kV", "Banco Baterias", "Cargadores", "SCADA", "RTU", "Medidores"], 5, 42, 0.10),
    ("3", "3. Suministros", "3.4", "3.4 Misceláneos", ["Cable Cobre Malla", "Cable XLPE", "Cable Control", "Conectores", "Terminales", "Bandejas Portacables"], 10, 45, 0.05),

    ("4", "4. Montaje", "4.1", "4.1 Malla a Tierra", ["Tendido Malla", "Soldaduras", "Colas Estructuras", "Medicion Resistencia", "Tratamiento Terreno"], 20, 30, 0.02),
    ("4", "4. Montaje", "4.2", "4.2 Equipos y Estructuras", ["Izaje Porticos", "Izaje Soportes", "Posicion Trafo", "Accesorios Trafo", "Montaje Interruptores", "Montaje Seccionadores", "Montaje TCs/TTs", "Alineacion", "Torqueo"], 30, 48, 0.05),
    ("4", "4. Montaje", "4.3", "4.3 Tendido y Conexionado", ["Potencia a Celdas", "Control a Edificio", "BT en Trafo", "Muflas", "Peinado Tableros", "Fibra Optica", "Bandejas"], 35, 50, 0.03),
    ("4", "4. Montaje", "4.4", "4.4 Salas Control", ["Fijacion Tableros", "Montaje GIS", "Banco Baterias", "Servicios Auxiliares", "Cableado Interpanel"], 38, 50, 0.02),

    ("5", "5. Pruebas", "5.1", "5.1 Pruebas SAT", ["Megado", "TTR", "Tiempos Cierre", "Resistencia Contactos", "Inyeccion Secundaria", "Descargas Parciales", "Rigidez Dielectrica"], 45, 51, 0.04),
    ("5", "5. Pruebas", "5.2", "5.2 Precomisionamiento", ["Wiring", "Logica Control", "Interbloqueos", "Señales SCADA", "Comunicaciones", "Lazos de Control"], 48, 52, 0.03),
    ("5", "5. Pruebas", "5.3", "5.3 Energización", ["Protocolos CND", "Inspeccion RETIE", "Energizacion Vacio", "Toma Carga", "Acta Entrega", "Capacitacion Operadores"], 51, 52, 0.01)
]

# Diccionarios de narrativa progresiva
nar_civ_op = ["Nivel freatico detectado. Inicia achique.", "Saturacion de terreno. 2 bombas extra.", "Achique continuo, frente a media marcha.", "Barro severo. Accesos limitados.", "Fundicion lenta por lodos en foso."]
nar_civ_adm = ["Costo directo elevado por ACPM.", "Aprobacion horas extras achique.", "Alquiler maquinaria impacta CPI.", "Mitigacion ambiental en curso.", "Alerta presupuestal en obra civil."]
nar_sum_op = ["Retraso en transito maritimo notificado.", "Inspeccion fisica aduanera en curso.", "Discrepancia documental en celdas.", "Equipos retenidos temporalmente.", "Reprogramando contratista de montaje."]
nar_sum_adm = ["Costos de bodegaje en puerto activos.", "Gestionando exencion de multas.", "SPI cae por no ingreso de equipos.", "Sobrecosto logistico acumulado.", "Retraso impacta facturacion del hito."]
nar_ok = ["Frente operativo.", "Ejecucion conforme."]

actividades = []
for cap_id, cap_name, sub_id, sub_name, lista_acts, ini, fin, peso_sub in wbs_base:
    peso_act = peso_sub / len(lista_acts)
    for i, act in enumerate(lista_acts):
        wbs_code = f"{sub_id}.{i+1}"
        actividades.append({
            "wbs": wbs_code, "cap": cap_name, "subcap": sub_name, "act": act,
            "ini": ini, "fin": fin, "bac": int(BAC_TOTAL * peso_act)
        })

# Corrección estricta: Inyección del remanente por redondeo
diff_bac = BAC_TOTAL - sum(a['bac'] for a in actividades)
actividades[0]['bac'] += diff_bac

# =====================================================================
# 3. MOTOR DE CÁLCULO EVM Y MATRIZ PLANA
# =====================================================================
rows = []
for act in actividades:
    pv_acc, ev_acc, ac_acc = 0, 0, 0
    duracion = act["fin"] - act["ini"] + 1
    pv_inc = act["bac"] / duracion

    for sem in range(1, PLAZO + 1):
        if act["ini"] <= sem <= act["fin"]: pv_acc = int(min(pv_acc + pv_inc, act["bac"]))
        elif sem > act["fin"]: pv_acc = act["bac"]

        coment_op, coment_adm = "", ""

        # Simulación hasta la Semana de Corte
        if sem <= SEM_CORTE:
            if sem >= act["ini"]:
                actual_spi = 0.96 if act["cap"].startswith("2") else (0.85 if act["cap"].startswith("3") else 1.0)
                actual_cpi = 0.76 if act["cap"].startswith("2") else (0.95 if act["cap"].startswith("3") else 1.0)

                if act["cap"].startswith("3") and sem >= 15: actual_spi = 0.65

                ev_inc_sem = pv_inc * actual_spi
                ac_inc_sem = ev_inc_sem / actual_cpi
                ev_acc = int(min(ev_acc + ev_inc_sem, act["bac"])) # Ensure EV does not exceed BAC for a single activity
                ac_acc = int(ac_acc + ac_inc_sem)

                if ev_acc >= act["bac"]:
                    coment_op, coment_adm = "Actividad terminada.", "Costo cerrado."
                else:
                    if act["cap"].startswith("2") and sem >= 12:
                        n_idx = (sem - 12) % len(nar_civ_op)
                        coment_op, coment_adm = nar_civ_op[n_idx], nar_civ_adm[n_idx]
                    elif act["cap"].startswith("3") and sem >= 15:
                        n_idx = (sem - 15) % len(nar_sum_op)
                        coment_op, coment_adm = nar_sum_op[n_idx], nar_sum_adm[n_idx]
                    else:
                        coment_op, coment_adm = nar_ok[0], nar_ok[1]
            else:
                ev_acc, ac_acc = 0, 0
                coment_op, coment_adm = "No iniciada", "No iniciada"

            rows.append([act["wbs"], act["cap"], act["subcap"], act["act"], sem, act["bac"], pv_acc, ev_acc, ac_acc, coment_op, coment_adm])
        else:
            # Semanas futuras (Para proyecciones PV)
            rows.append([act["wbs"], act["cap"], act["subcap"], act["act"], sem, act["bac"], pv_acc, "", "", "", ""])

df = pd.DataFrame(rows, columns=['WBS', 'Capitulo', 'Subcapitulo', 'Actividad', 'Semana', 'BAC', 'PV', 'EV', 'AC', 'Comentario_Operativo', 'Comentario_Administrativo'])

# =====================================================================
# 4. CALIBRACIÓN ABSOLUTA EN SEMANA 20
# =====================================================================
mask_20 = df["Semana"] == SEM_CORTE
diff_pv = TARGET_PV - df.loc[mask_20, "PV"].sum()
diff_ev = TARGET_EV - pd.to_numeric(df.loc[mask_20, "EV"]).sum()
diff_ac = TARGET_AC - pd.to_numeric(df.loc[mask_20, "AC"]).sum()

idx_activos = df[(mask_20) & (df["Capitulo"].str.startswith("2")) & (df["EV"] != 0)].index
if len(idx_activos) > 0:
    act_name = df.loc[idx_activos[0], "Actividad"]
    # Ajustar valor acumulado de esa actividad desde la semana de corte en adelante
    mask_ajuste = (df["Actividad"] == act_name) & (df["Semana"] == SEM_CORTE)
    df.loc[mask_ajuste, "PV"] += diff_pv
    df.loc[mask_ajuste, "EV"] = pd.to_numeric(df.loc[mask_ajuste, "EV"]) + diff_ev
    df.loc[mask_ajuste, "AC"] = pd.to_numeric(df.loc[mask_ajuste, "AC"]) + diff_ac

# Exportación del CSV (Solo datos históricos hasta la semana de corte para la matriz plana)
df_export = df[df["Semana"] <= SEM_CORTE].copy()
df_export.to_csv('data_proyecto.csv', sep=';', index=False)

# Auditoría Consola
df_20_final = df_export[df_export["Semana"] == SEM_CORTE].copy() # Fixed: Use df_export['Semana']
cpi = round(df_20_final["EV"].sum() / df_20_final["AC"].sum(), 2)
spi = round(df_20_final["EV"].sum() / df_20_final["PV"].sum(), 2)
print(f"--- SIMULACIÓN EJECUTADA ---")
print(f"BAC Total: {df_20_final['BAC'].sum()}")
print(f"Auditoría Sem 20 -> CPI: {cpi} | SPI: {spi}")
print(f"Archivo 'matriz_control_subestacion.csv' exportado.")

# =====================================================================
# 5. RENDERIZADO DE CURVAS S EN PANTALLA (MATPLOTLIB)
# =====================================================================
df['PV'] = pd.to_numeric(df['PV'], errors='coerce')
df['EV'] = pd.to_numeric(df['EV'], errors='coerce')
df['AC'] = pd.to_numeric(df['AC'], errors='coerce')

df_plot = df.groupby('Semana').agg({'PV': 'sum', 'EV': 'sum', 'AC': 'sum'}).reset_index()

# Ocultar ceros futuros para EV y AC en la gráfica
df_plot.loc[df_plot['Semana'] > SEM_CORTE, ['EV', 'AC']] = np.nan

plt.figure(figsize=(12, 6))
plt.plot(df_plot['Semana'], df_plot['PV'], label='Valor Planificado (PV)', color='#1f77b4', linestyle='--', linewidth=2)
plt.plot(df_plot['Semana'], df_plot['EV'], label='Valor Ganado (EV)', color='#2ca02c', linewidth=2.5)
plt.plot(df_plot['Semana'], df_plot['AC'], label='Costo Real (AC)', color='#d62728', linewidth=2.5)

plt.axvline(x=SEM_CORTE, color='gray', linestyle=':', label=f'Corte (Sem {SEM_CORTE})')

# Add labels to the end of the lines
last_sem = df_plot['Semana'].max()

# Label for PV with value
last_pv = df_plot.loc[df_plot['Semana'] == last_sem, 'PV'].iloc[0]
plt.text(last_sem - 0.5, last_pv, f'PV: ${last_pv:,.0f}', color='#1f77b4', va='center', ha='right', fontsize=10, fontweight='bold')

# Label for EV (at SEM_CORTE) with value
last_ev_sem_corte = df_plot.loc[df_plot['Semana'] == SEM_CORTE, 'EV'].iloc[0]
plt.text(SEM_CORTE + 0.5, last_ev_sem_corte, f'EV: ${last_ev_sem_corte:,.0f}', color='#2ca02c', va='center', ha='left', fontsize=10, fontweight='bold')

# Label for AC (at SEM_CORTE) with value
last_ac_sem_corte = df_plot.loc[df_plot['Semana'] == SEM_CORTE, 'AC'].iloc[0]
plt.text(SEM_CORTE + 0.5, last_ac_sem_corte - 0.05 * (df_plot['PV'].max()), f'AC: ${last_ac_sem_corte:,.0f}', color='#d62728', va='center', ha='left', fontsize=10, fontweight='bold')

# Label for the cut-off line
plt.text(SEM_CORTE, plt.ylim()[1] * 0.95, f'Corte (Sem {SEM_CORTE})', color='gray', ha='center', va='top', fontsize=10)

plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['AC'], where=(df_plot['Semana'] <= SEM_CORTE), color='red', alpha=0.1, label='Sobrecosto (Variación)')
plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['PV'], where=(df_plot['Semana'] <= SEM_CORTE), color='orange', alpha=0.1, label='Retraso (Variación)')

plt.title('Curva S de Desempeño - Subestación 115kV (Simulación)', fontsize=14, fontweight='bold')
plt.xlabel('Semanas del Proyecto', fontsize=11)
plt.ylabel('Inversión Acumulada (USD)', fontsize=11)
plt.xticks(np.arange(0, 53, step=4))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc='upper left')
plt.tight_layout()

# Mostrar Curvas
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from fpdf import FPDF
import sqlite3
import matplotlib.ticker as mticker

def procesar_dataset_simulado(archivo_csv):
    """Carga y limpia el dataset, transformando el formato regional a numérico.

    Args:
        archivo_csv (str): La ruta al archivo CSV simulado.

    Returns:
        pd.DataFrame: El DataFrame procesado con los datos limpios.

    Raises:
        FileNotFoundError: Si no se encontró el archivo CSV.
    """
    if not os.path.exists(archivo_csv):
        raise FileNotFoundError(f"No se encontró el archivo simulado: {archivo_csv}. Ejecute el script generador primero.")

    df = pd.read_csv(archivo_csv, sep=';')
    # print("Columnas del DataFrame cargado:", df.columns.tolist()) # Línea de depuración para verificar nombres de columnas

    # Limpieza de formato numérico (de '1234,56' a 1234.56) para columnas de moneda
    cols_moneda = ["BAC", "PV", "EV", "AC"]
    for col in cols_moneda:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].str.replace(',', '.').astype(float)
        elif col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # CORRECCIÓN DE ESCALADO PARA PV, EV, AC
    # Heurística: Si los valores máximos de PV, EV o AC son mucho mayores que el BAC total,
    # asumimos un error de escalado por 100 y corregimos.
    if 'BAC' in df.columns and 'PV' in df.columns and 'EV' in df.columns and 'AC' in df.columns:
        # Calcular el BAC total único del proyecto para la heurística
        # Agregamos 'WBS' a la selección para asegurar que el drop_duplicates funcione correctamente en el contexto
        # de BAC por WBS, que es lo que queremos sumar para el BAC total del proyecto.
        total_bac_unique = df[['WBS', 'BAC']].drop_duplicates(subset=['WBS'])['BAC'].sum()

        # Si el BAC total es cero, o si hay NaNs, esta heurística podría fallar o no ser aplicable.
        # Procede solo si BAC total es > 0.
        # The scaling correction block has been removed as per the subtask.

    # Debugging prints for PV, EV, AC at SEM_CORTE after loading
    sem_corte_local = globals().get('SEM_CORTE', 20)
    df_filtered_at_corte = df[df['Semana'] == sem_corte_local]

    if not df_filtered_at_corte.empty:
        pv_sum_loaded = df_filtered_at_corte['PV'].sum()
        ev_sum_loaded = df_filtered_at_corte['EV'].sum()
        ac_sum_loaded = df_filtered_at_corte['AC'].sum()
        print(f"[DEBUG - Carga de Datos] Sumas en Semana {sem_corte_local}: PV={pv_sum_loaded:,.0f}, EV={ev_sum_loaded:,.0f}, AC={ac_sum_loaded:,.0f}")
    else:
        print(f"[DEBUG - Carga de Datos] No hay datos para la Semana {sem_corte_local} en df_loaded.")

    return df

def calcular_indicadores_por_capitulo(df, sem_corte):
    """Calcula el CPI y SPI agregado por frente de obra constructivo.

    Args:
        df (pd.DataFrame): El DataFrame de proyecto con los datos cargados.
        sem_corte (int): La semana de corte para el análisis.

    Returns:
        pd.DataFrame: Un DataFrame con los KPIs de valor ganado calculados por capítulo.
    """
    # Calculate BAC per chapter from the full dataset, considering unique WBS BACs
    df_unique_wbs_bac = df[['WBS', 'Capitulo', 'BAC']].drop_duplicates(subset=['WBS'])
    bac_por_capitulo = df_unique_wbs_bac.groupby('Capitulo')['BAC'].sum().reset_index()
    bac_por_capitulo = bac_por_capitulo.rename(columns={'BAC': 'BAC_Capitulo_Total'})

    # FIX: Filter for the SEM_CORTE only for PV, EV, AC calculation
    # The previous code filtered for Semana <= sem_corte and then summed, which is incorrect for cumulative totals at a specific point in time.
    df_filtered_at_corte_for_kpis = df[df['Semana'] == sem_corte].copy()
    resumen = df_filtered_at_corte_for_kpis.groupby('Capitulo')[["PV", "EV", "AC"]].sum().reset_index()

    # Merge the calculated BAC_Capitulo_Total into the resumen DataFrame
    resumen = pd.merge(resumen, bac_por_capitulo, on='Capitulo', how='left')
    resumen = resumen.rename(columns={'BAC_Capitulo_Total': 'BAC'})

    resumen['CPI'] = np.where(resumen['AC'] == 0, 0, resumen['EV'] / resumen['AC'])
    resumen['SPI'] = np.where(resumen['PV'] == 0, 0, resumen['EV'] / resumen['PV'])
    return resumen

# =============================================================================
# Funciones de Creación y Población de Base de Datos SQLite
# =============================================================================

def create_and_populate_sqlite_db(df_source, db_file):
    """Crea el archivo de la base de datos SQLite, define los esquemas de las tablas
    y carga los datos desde df_source en las tablas correspondientes.
    """
    # Eliminar el archivo de la base de datos si ya existe para empezar de cero
    if os.path.exists(db_file):
        os.remove(db_file)
        print(f"Base de datos existente '{db_file}' eliminada.")

    conn = None
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        print(f"Base de datos '{db_file}' creada exitosamente.")

        # 2. Crear la tabla Dim_WBS
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Dim_WBS (
                WBS TEXT PRIMARY KEY,
                Capitulo TEXT,
                Subcapitulo TEXT,
                Actividad TEXT
            )
        """
        )
        print("Tabla 'Dim_WBS' creada.")

        # 3. Crear la tabla Fact_Seguimiento
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Fact_Seguimiento (
                WBS TEXT,
                Semana INTEGER,
                EV INTEGER,
                AC INTEGER,
                Comentario_Operativo TEXT,
                Comentario_Administrativo TEXT,
                FOREIGN KEY (WBS) REFERENCES Dim_WBS(WBS)
            )
        """
        )
        print("Tabla 'Fact_Seguimiento' creada.")

        # 4. Crear la tabla Fact_Proyecto
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Fact_Proyecto (
                WBS TEXT,
                Semana INTEGER,
                PV INTEGER,
                BAC INTEGER,
                FOREIGN KEY (WBS) REFERENCES Dim_WBS(WBS)
            )
        """
        )
        print("Tabla 'Fact_Proyecto' creada.")

        # Extraer datos y cargarlos
        df_dim_wbs = df_source[['WBS', 'Capitulo', 'Subcapitulo', 'Actividad']].drop_duplicates().copy()
        df_fact_seguimiento = df_source[['WBS', 'Semana', 'EV', 'AC', 'Comentario_Operativo', 'Comentario_Administrativo']].copy()
        df_fact_proyecto = df_source[['WBS', 'Semana', 'PV', 'BAC']].copy()

        df_dim_wbs.to_sql('Dim_WBS', conn, if_exists='append', index=False)
        df_fact_seguimiento.to_sql('Fact_Seguimiento', conn, if_exists='append', index=False)
        df_fact_proyecto.to_sql('Fact_Proyecto', conn, if_exists='append', index=False)

        conn.commit()
        print("Datos cargados en las tablas SQLite.")

    except sqlite3.Error as e:
        print(f"Error al crear o poblar la base de datos: {e}")
    finally:
        if conn:
            conn.close()
            print("Conexión a la base de datos cerrada.")

def load_dimensional_model_from_sqlite(db_file):
    """Carga el modelo dimensional completo desde la base de datos SQLite.

    Returns:
        pd.DataFrame: El DataFrame combinado con todos los datos.
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        df_dim_wbs = pd.read_sql_query("SELECT * FROM Dim_WBS", conn)
        df_fact_seguimiento = pd.read_sql_query("SELECT * FROM Fact_Seguimiento", conn)
        df_fact_proyecto = pd.read_sql_query("SELECT * FROM Fact_Proyecto", conn)

        df_merged_facts = pd.merge(df_fact_seguimiento, df_fact_proyecto,
                                 on=['WBS', 'Semana'],
                                 how='inner')
        df_modelo_dimensional = pd.merge(df_merged_facts, df_dim_wbs,
                                       on='WBS',
                                       how='inner')
        return df_modelo_dimensional
    except sqlite3.Error as e:
        print(f"Error al cargar el modelo dimensional desde SQLite: {e}")
        return pd.DataFrame()
    finally:
        if conn:
            conn.close()

# =============================================================================
# Funciones de Graficado
# =============================================================================

def graficar_curva_s_dimensional(df_input, semana_corte, filename='s_curve_plot.png'):
    """Renderiza la Curva S leyendo directamente la proyección de la Fact Table.
    Utiliza los datos de PV, EV, AC ya presentes y agrega por semana.
    """
    df_plot = df_input.groupby('Semana').agg({'PV': 'sum', 'EV': 'sum', 'AC': 'sum'}).reset_index()

    # Ocultar futuros para no distorsionar la gráfica
    df_plot.loc[df_plot['Semana'] > semana_corte, ['EV', 'AC']] = np.nan

    plt.figure(figsize=(8, 6)) # Updated figsize
    plt.plot(df_plot['Semana'], df_plot['PV'], label='Valor Planificado (PV)', color='#1f77b4', linestyle='--', linewidth=2)
    plt.plot(df_plot['Semana'], df_plot['EV'], label='Valor Ganado (EV)', color='#2ca02c', linewidth=2.5)
    plt.plot(df_plot['Semana'], df_plot['AC'], label='Costo Real (AC)', color='#d62728', linewidth=2.5)

    plt.axvline(x=semana_corte, color='grey', linestyle=':', label=f'Corte S{semana_corte}')

    # Set x-axis ticks to integers
    plt.gca().xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Anotaciones
    pv_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'PV'].values
    ev_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'EV'].values
    ac_values = df_plot.loc[df_plot['Semana'] == semana_corte, 'AC'].values

    pv_at_cut = pv_values[0] if pv_values.size > 0 else 0
    ev_at_cut = ev_values[0] if ev_values.size > 0 else 0
    ac_at_cut = ac_values[0] if ac_values.size > 0 else 0
    max_y = df_plot['PV'].max() if not df_plot['PV'].empty else 1 # Avoid division by zero if empty

    # Adjusted PV label position
    plt.text(semana_corte + 0.5, pv_at_cut - 0.005*max_y, f'PV: ${pv_at_cut:,.0f}', color='#1f77b4', fontsize=11)
    plt.text(semana_corte + 0.5, ev_at_cut + 0.005*max_y, f'EV: ${ev_at_cut:,.0f}', color='#2ca02c', fontsize=11)
    plt.text(semana_corte + 0.5, ac_at_cut - 0.02*max_y, f'AC: ${ac_at_cut:,.0f}', color='#d62728', fontsize=11)

    # Zonas de brecha (Relleno)
    plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['PV'], where=(df_plot['Semana'] <= semana_corte) & (df_plot['EV'] < df_plot['PV']), color='orange', alpha=0.1, label='Retraso')
    plt.fill_between(df_plot['Semana'], df_plot['EV'], df_plot['AC'], where=(df_plot['Semana'] <= semana_corte) & (df_plot['EV'] < df_plot['AC']), color='red', alpha=0.1, label='Sobrecosto')


    plt.title('Curva S de Avance de Proyecto', fontsize=16, fontweight='bold')
    plt.xlabel('Semana del Proyecto', fontsize=12)
    plt.ylabel('Inversión Acumulada ($)', fontsize=12)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

def graficar_matriz_desempeno(chapter_kpis_df, filename='temp_matriz_cpispi.png'):
    """Genera y guarda el gráfico de matriz de desempeño CPI vs SPI.
    """
    cpi_vals = chapter_kpis_df['CPI'].tolist()
    spi_vals = chapter_kpis_df['SPI'].tolist()
    categorias = chapter_kpis_df['Capitulo'].tolist()

    plt.figure(figsize=(8, 6)) # Updated figsize
    plt.scatter(spi_vals, cpi_vals, color='navy', s=150, zorder=5)
    plt.axhline(1.0, color='red', linestyle='--', alpha=0.5)
    plt.axvline(1.0, color='red', linestyle='--', alpha=0.5)

    plt.fill_between([0, 1], 0, 1, color='red', alpha=0.1)
    plt.fill_between([1, 1.5], 1, 1.5, color='green', alpha=0.1)

    plt.xlim(0.5, 1.2)
    plt.ylim(0.5, 1.2)
    plt.xlabel('SPI (Eficiencia Cronograma)', fontsize=12)
    plt.ylabel('CPI (Eficiencia Costo)', fontsize=12)
    plt.title('Matriz de Desempeño (CPI vs SPI)', fontsize=16, fontweight='bold')

    for i, txt in enumerate(categorias):
        plt.annotate(txt, (spi_vals[i], cpi_vals[i]), xytext=(5,5), textcoords='offset points', fontsize=10)

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

# =============================================================================
# Funciones de Generación de Informes PDF
# =============================================================================

class InformePMO(FPDF):
    """Clase personalizada para generar informes PDF con encabezados y pies de página específicos.
    """
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.set_text_color(0, 51, 102)
        self.cell(0, 10, 'PROJECT MANAGEMENT OFFICE (PMO)', border=0, ln=1, align='C')
        self.set_font('Arial', 'I', 11)
        self.set_text_color(128, 128, 128)
        self.cell(0, 5, 'Informe Ejecutivo de Estado de Proyecto - Arquitectura Dimensional', border='B', ln=1, align='C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(128, 128, 128)
        self.set_x(10)
        self.cell(0, 10, 'Fuente: Datos consolidados de proyecto', 0, 0, 'L')
        self.set_x(self.w / 2 - 25)
        self.cell(0, 10, 'Confidencial - Uso Interno', 0, 0, 'C')
        self.set_x(self.w - 20)
        self.cell(0, 10, f'Página {self.page_no()}/{{nb}}', 0, 0, 'R')

    def titulo_seccion(self, titulo):
        self.set_font('Arial', 'B', 10)
        self.set_text_color(255, 255, 255)
        self.set_fill_color(0, 51, 102)
        self.cell(0, 8, f'  {titulo}', border=0, ln=1, align='L', fill=True)
        self.ln(4)

    def texto_normal(self, texto):
        self.set_font('Arial', '', 10)
        self.set_text_color(0, 0, 0)
        self.multi_cell(0, 5, texto)
        self.ln(2)

def generar_pdf_ejecutivo(df_master, chapter_kpis_df, semana_corte):
    """Ensambla el documento PDF con la narrativa y los gráficos generados.
    """
    # Ensure BAC_TOTAL is retrieved from globals
    # global_bac_total = globals().get('BAC_TOTAL', 0) # This line is replaced

    graficar_curva_s_dimensional(df_master, semana_corte)
    graficar_matriz_desempeno(chapter_kpis_df)

    pdf = InformePMO()
    pdf.alias_nb_pages()
    pdf.add_page()

    # Metadatos
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(40, 6, 'Proyecto:', 0, 0)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 6, 'Construcción Subestación Eléctrica 115kV/34.5kV', 0, 1)

    pdf.set_font('Arial', 'B', 12)
    pdf.cell(40, 6, 'Fecha de Corte:', 0, 0)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 6, f'Semana {semana_corte} ({datetime.now().strftime("%Y-%m-%d")})', 0, 1)
    pdf.ln(5)

    # Sección 1
    pdf.titulo_seccion('1. RESUMEN EJECUTIVO Y MÉTRICAS GLOBALES')
    texto_resumen = (
        "El proyecto presenta un Índice de Desempeño de Costo (CPI) de 0.82, indicando un "
        "sobrecosto estructural originado en la fase de obras civiles. Simultáneamente, el Índice "
        "de Desempeño de Cronograma (SPI) se sitúa en ~0.86, reflejando un retraso en la ruta "
        "crítica debido a contingencias aduaneras en la importación de suministros mayores."
    )
    pdf.texto_normal(texto_resumen)

    # Tabla KPIs
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 8, 'Desempeño de KPIs por Capítulo', 0, 1, 'C')
    pdf.ln(2)

    col_widths = [40, 20, 20, 20, 20, 15, 15] # Adjusted 'Capitulo' width from 75 to 90
    headers = ['Capítulo', 'BAC', 'PV', 'EV', 'AC', 'CPI', 'SPI']
    total_table_width = sum(col_widths)
    start_x = (pdf.w - total_table_width) / 2

    pdf.set_x(start_x)
    pdf.set_fill_color(220, 220, 220)
    pdf.set_text_color(0, 0, 0)
    for i, header in enumerate(headers):
        pdf.cell(col_widths[i], 8, header, 1, 0, 'C', 1)
    pdf.ln()

    pdf.set_font('Arial', '', 9)
    for _, row in chapter_kpis_df.iterrows():
        pdf.set_x(start_x)
        pdf.cell(col_widths[0], 7, row['Capitulo'], 1, 0, 'L')
        pdf.cell(col_widths[1], 7, f"{row['BAC']:,.0f}", 1, 0, 'R')
        pdf.cell(col_widths[2], 7, f"{row['PV']:,.0f}", 1, 0, 'R')
        pdf.cell(col_widths[3], 7, f"{row['EV']:,.0f}", 1, 0, 'R')
        pdf.cell(col_widths[4], 7, f"{row['AC']:,.0f}", 1, 0, 'R')
        pdf.cell(col_widths[5], 7, f"{row['CPI']:.2f}", 1, 0, 'R')
        pdf.cell(col_widths[6], 7, f"{row['SPI']:.2f}", 1, 0, 'R')
        pdf.ln()

    # Totales Globales
    # FIX: Calculate total PV, EV, AC by summing cumulative values at SEM_CORTE, not all weekly entries.
    df_at_corte_final = df_master[df_master['Semana'] == semana_corte]
    total_pv = df_at_corte_final['PV'].sum()
    total_ev = df_at_corte_final['EV'].sum()
    total_ac = df_at_corte_final['AC'].sum()

    global_cpi = total_ev / total_ac if total_ac != 0 else 0
    global_spi = total_ev / total_pv if total_pv != 0 else 0

    # Use the global BAC_TOTAL variable
    global_bac_total = BAC_TOTAL

    # Añadir depuración en generación de PDF
    TARGET_PV_GLOBAL = globals().get('TARGET_PV', 0)
    TARGET_EV_GLOBAL = globals().get('TARGET_EV', 0)
    TARGET_AC_GLOBAL = globals().get('TARGET_AC', 0)

    print(f"\n[DEBUG - Generación PDF] Total PV global para Curva S y tabla: {total_pv:,.0f}")
    print(f"[DEBUG - Generación PDF] Total EV global para Curva S y tabla: {total_ev:,.0f}")
    print(f"[DEBUG - Generación PDF] Total AC global para Curva S y tabla: {total_ac:,.0f}")
    print(f"[DEBUG - Generación PDF] TARGET PV (global): {TARGET_PV_GLOBAL:,.0f} | Diferencia: {total_pv - TARGET_PV_GLOBAL:,.0f}")
    print(f"[DEBUG - Generación PDF] TARGET EV (global): {TARGET_EV_GLOBAL:,.0f} | Diferencia: {total_ev - TARGET_EV_GLOBAL:,.0f}")
    print(f"[DEBUG - Generación PDF] TARGET AC (global): {TARGET_AC_GLOBAL:,.0f} | Diferencia: {total_ac - TARGET_AC_GLOBAL:,.0f}")


    pdf.set_x(start_x)
    pdf.set_fill_color(200, 200, 200)
    pdf.set_font('Arial', 'B', 9)
    pdf.cell(col_widths[0], 7, 'Totales', 1, 0, 'L', 1)
    pdf.cell(col_widths[1], 7, f"{global_bac_total:,.0f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[2], 7, f"{total_pv:,.0f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[3], 7, f"{total_ev:,.0f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[4], 7, f"{total_ac:,.0f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[5], 7, f"{global_cpi:.2f}", 1, 0, 'R', 1)
    pdf.cell(col_widths[6], 7, f"{global_spi:.2f}", 1, 0, 'R', 1)
    pdf.ln(10)

    # --- Sección 2: ANÁLISIS TÉCNICO POR FRENTE CONSTRUCTIVO ---
    pdf.titulo_seccion('2. ANÁLISIS TÉCNICO POR FRENTE CONSTRUCTIVO')

    # Extract CPI and SPI values for each chapter with error handling
    try:
        ingenieria_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '1. Ingeniería'].iloc[0]
        ing_cpi = ingenieria_kpis['CPI']
        ing_spi = ingenieria_kpis['SPI']
    except IndexError:
        ing_cpi = 0.0
        ing_spi = 0.0

    try:
        obras_civiles_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '2. Obras Civiles'].iloc[0]
        oc_cpi = obras_civiles_kpis['CPI']
        oc_spi = obras_civiles_kpis['SPI']
    except IndexError:
        oc_cpi = 0.0
        oc_spi = 0.0

    try:
        suministros_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '3. Suministros'].iloc[0]
        sum_cpi = suministros_kpis['CPI']
        sum_spi = suministros_kpis['SPI']
    except IndexError:
        sum_cpi = 0.0
        sum_spi = 0.0

    try:
        montaje_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '4. Montaje'].iloc[0]
        mon_cpi = montaje_kpis['CPI']
        mon_spi = montaje_kpis['SPI']
    except IndexError:
        mon_cpi = 0.0
        mon_spi = 0.0

    try:
        pruebas_kpis = chapter_kpis_df[chapter_kpis_df['Capitulo'] == '5. Pruebas'].iloc[0]
        prue_cpi = pruebas_kpis['CPI']
        prue_spi = pruebas_kpis['SPI']
    except IndexError:
        prue_cpi = 0.0
        prue_spi = 0.0

    # Insert dynamically generated narrative
    texto_ingenieria = (
        f"INGENIERÍA (CPI: {ing_cpi:.2f} | SPI: {ing_spi:.2f}): La fase de ingeniería ha progresado según lo planeado en cronograma, "
        "pero ha incurrido en costos adicionales, principalmente debido a estudios "
        "complementarios no previstos y ajustes en diseños de detalle."
    )
    pdf.texto_normal(texto_ingenieria)

    texto_civil = (
        f"OBRAS CIVILES (CPI: {oc_cpi:.2f} | SPI: {oc_spi:.2f}): Impacto financiero crítico. Se detectó un nivel "
        "freático excepcionalmente alto durante excavaciones de pórticos. Esto obligó al uso "
        "continuo de motobombas y entibados, erosionando las reservas de gestión de este paquete."
    )
    pdf.texto_normal(texto_civil)

    texto_suministros = (
        f"SUMINISTROS DE POTENCIA (CPI: {sum_cpi:.2f} | SPI: {sum_spi:.2f}): Retraso severo. Las órdenes de compra "
        "del Transformador de 30MVA y celdas GIS presentan demoras por huelgas portuarias e "
        "inspecciones aduaneras, desplazando los hitos FAT."
    )
    pdf.texto_normal(texto_suministros)

    texto_montaje = (
        f"MONTAJE (CPI: {mon_cpi:.2f} | SPI: {mon_spi:.2f}): La fase de montaje ha mantenido un buen ritmo, "
        "pero la disponibilidad de algunos equipos menores y la espera por la llegada de los "
        "suministros críticos podría impactar el cronograma futuro."
    )
    pdf.texto_normal(texto_montaje)

    texto_pruebas = (
        f"PRUEBAS (CPI: {prue_cpi:.2f} | SPI: {prue_spi:.2f}): Esta fase aún no ha iniciado actividades con costo significativo, "
        "manteniendo sus indicadores en cero. El inicio dependerá de la finalización de montaje y la disponibilidad de equipos."
    )
    pdf.texto_normal(texto_pruebas)

# Inserción de Gráfico de Cuadrantes (Salud del Proyecto) y Curva S lado a lado
    y_actual = pdf.get_y()

    # Ajuste: Se resta 2mm a la posición actual para subir los gráficos 3mm respecto al +1 anterior
    y_posicion_graficos = y_actual - 2

    # Calculate width for two plots with margins and spacing
    plot_width = 87.5  # (210 - (15*2) - 5) / 2 = 87.5mm

    # Se aplica y_posicion_graficos para elevar ambas imágenes
    pdf.image('temp_matriz_cpispi.png', x=15, y=y_posicion_graficos, w=plot_width)
    pdf.image('s_curve_plot.png', x=15 + plot_width + 5, y=y_posicion_graficos, w=plot_width)

    # Ajuste del salto de línea para compensar el movimiento hacia arriba de los gráficos
    pdf.ln(65)

    # Exportación
    pdf_filename = f"Informe_Direccion_Semana_{semana_corte}.pdf"
    pdf.output(pdf_filename)
    print(f"Informe ejecutivo generado exitosamente: {pdf_filename}")

# =============================================================================
# BLOQUE DE EJECUCIÓN PRINCIPAL (CON DEPENDENCIA ESTRICTA DE CSV)
# =============================================================================
if __name__ == "__main__":
    # 1. Configuración de nombres de archivos
    archivo_csv_datos = 'data_proyecto.csv'
    db_file = 'project_data.db'

    # 2. Validación de Seguridad: Si no existe el CSV, el script DEBE FALLAR
    if not os.path.exists(archivo_csv_datos):
        print(f"\n[ERROR CRÍTICO]: No se encontró el archivo fuente '{archivo_csv_datos}'.")
        print("La política de integridad exige este archivo para generar el reporte.")
        raise FileNotFoundError(f"Falta el archivo obligatorio: {archivo_csv_datos}")

    # 3. Recuperación de variables de control
    # Se obtienen de globals si el script de simulación corrió en la misma sesión,
    # de lo contrario se usan los estándares del proyecto.
    SEM_CORTE = globals().get('SEM_CORTE', 20)

    print(f"\n--- INICIANDO PROCESAMIENTO SEMANA {SEM_CORTE} ---")

    # 4. Fase ETL: Carga y Limpieza desde CSV
    print(f"Cargando datos obligatorios desde: {archivo_csv_datos}...")
    df_loaded = procesar_dataset_simulado(archivo_csv_datos)

    # 5. Persistencia: Poblado de Base de Datos SQLite
    print(f"Poblando base de datos relacional '{db_file}' para análisis dimensional...")
    create_and_populate_sqlite_db(df_loaded, db_file)

    # 6. Análisis: Carga del Modelo Dimensional desde SQL
    df_modelo_dimensional = load_dimensional_model_from_sqlite(db_file)

    # Añadir depuración en modelo dimensional: Mostrar suma de PV, EV, AC en SEM_CORTE
    sem_corte_main = globals().get('SEM_CORTE', 20)
    df_model_at_sem_corte = df_modelo_dimensional[df_modelo_dimensional['Semana'] == sem_corte_main]

    if not df_model_at_sem_corte.empty:
        pv_sum_model = df_model_at_sem_corte['PV'].sum()
        ev_sum_model = df_model_at_sem_corte['EV'].sum()
        ac_sum_model = df_model_at_sem_corte['AC'].sum()
        print(f"\n[DEBUG - Modelo Dimensional] Suma de PV en Sem {sem_corte_main} (df_modelo_dimensional): {pv_sum_model:,.0f}")
        print(f"[DEBUG - Modelo Dimensional] Suma de EV en Sem {sem_corte_main} (df_modelo_dimensional): {ev_sum_model:,.0f}")
        print(f"[DEBUG - Modelo Dimensional] Suma de AC en Sem {sem_corte_main} (df_modelo_dimensional): {ac_sum_model:,.0f}")
    else:
        print(f"\n[DEBUG - Modelo Dimensional] No hay datos para la Semana {sem_corte_main} en df_modelo_dimensional.")


    # Calculate BAC_TOTAL globally from df_modelo_dimensional
    BAC_TOTAL = df_modelo_dimensional[['WBS', 'BAC']].drop_duplicates(subset=['WBS'])['BAC'].sum()

    # 7. Cálculo de Inteligencia de Negocios (KPIs)
    chapter_kpis_df = calcular_indicadores_por_capitulo(df_modelo_dimensional, SEM_CORTE)

    print("\nResumen de Desempeño por Capítulo (Auditoría Técnica):")
    print(chapter_kpis_df[['Capitulo', 'BAC', 'PV', 'EV', 'AC', 'CPI', 'SPI']].round(2))

    # 8. Generación del Entregable PDF
    print("\nGenerando informe ejecutivo PDF...")
    generar_pdf_ejecutivo(df_modelo_dimensional, chapter_kpis_df, SEM_CORTE)

    # 9. Higiene del Directorio (Limpieza de temporales)
    for tmp_file in ['s_curve_plot.png', 'temp_matriz_cpispi.png']:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

    print(f"\n--- PROCESO FINALIZADO: Informe_Direccion_Semana_{SEM_CORTE}.pdf generado ---")